## Step 3 — Trunk Segmentation

This notebook performs trunk segmentation using the bounding boxes selected during the object detection step. Bounding boxes from the YOLO annotation file are used as prompts for the SAM2 segmentation model, which generates pixel-level masks for the detected trunk region in each image.

The workflow performs the following tasks:

1. Load the RGB images and YOLO bounding box annotations.
2. Initialize the SAM2 segmentation model.
3. Use each YOLO bounding box as a prompt for SAM2.
4. Generate binary trunk masks for each image.
5. Save segmentation masks in JSON format for downstream analysis.
6. Optionally overlay masks on the original images for visual inspection.

These segmentation masks are used in later steps to isolate trunk pixels and estimate trunk diameter from the corresponding depth data.

In [1]:
# Import libraries

import os
import gc
import math
import random
from datetime import datetime

import cv2
import ujson as json
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from tqdm import tqdm
import supervision as sv

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

## Set device and load dataset

In [ ]:
HOME = os.getcwd()
print("HOME:", HOME)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

if torch.cuda.is_available():
    print("CUDA available:", torch.cuda.is_available())
    print("Current device index:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("No CUDA-enabled GPU found. Using CPU.")

# Data paths
images_directory_path = "path/to/image_folder"
annotations_path = "path/to/yolo_annotations.json"

object_detection_dataset = sv.DetectionDataset.from_coco(
    images_directory_path=images_directory_path,
    annotations_path=annotations_path,
)

## Load SAM2 model and define segmentation function

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = "path/to/sam2_checkpoint.pt"
model_cfg = "path/to/sam2_config.yaml"

sam_predictor = SAM2ImagePredictor(
    build_sam2(model_cfg, checkpoint).to(device=DEVICE)
)


def segment(sam_predictor: SAM2ImagePredictor, image: np.ndarray, xyxy: np.ndarray) -> np.ndarray:
    sam_predictor.set_image(image)

    result_masks = []

    for box in xyxy:
        masks, scores, logits = sam_predictor.predict(
            box=box,
            multimask_output=True
        )

        index = np.argmax(scores)
        result_masks.append(masks[index])

    return np.array(result_masks)

## Run batch segmentation and save masks to JSON

In [ ]:
batch_size = 1000

images_directory_path = "path/to/image_folder"
json_filename_template = os.path.join(
    images_directory_path,
    "SAM2_mask_data_batch_{batch_number}.json"
)

annotation_lookup = object_detection_dataset.annotations

for batch_number, batch_start in enumerate(range(0, len(annotation_lookup), batch_size)):
    json_filename = json_filename_template.format(batch_number=batch_number)

    with open(json_filename, "w") as json_file:
        json_file.write("{\n")

        batch_annotations = list(annotation_lookup.items())[batch_start:batch_start + batch_size]

        for idx, (image_name, detections) in enumerate(
            tqdm(batch_annotations, desc=f"Processing batch {batch_number}")
        ):
            try:
                image = object_detection_dataset._get_image(image_name)

                detections.mask = segment(
                    sam_predictor=sam_predictor,
                    image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
                    xyxy=detections.xyxy
                )

                if detections.mask is not None:
                    shortened_image_name = os.path.splitext(os.path.basename(image_name))[0] + ".png"
                    mask_int = detections.mask.astype("uint8") * 255
                    mask_entry = '"{}": {}'.format(
                        shortened_image_name,
                        json.dumps(mask_int.tolist())
                    )

                    if idx < len(batch_annotations) - 1:
                        json_file.write(mask_entry + ",\n")
                    else:
                        json_file.write(mask_entry + "\n")

            except Exception as e:
                print(f"Error processing {image_name}: {e}")

            del image
            del detections.mask
            torch.cuda.empty_cache()
            gc.collect()

        json_file.write("}\n")

    torch.cuda.empty_cache()
    gc.collect()
    print(f"Batch {batch_number} saved to {json_filename}")

## Optional utility to fix backslashes in a JSON file

In [ ]:
def fix_json_backslashes(json_filename):
    with open(json_filename, "r") as f:
        content = f.read()

    fixed_content = content.replace("\\", "\\\\")

    directory, original_filename = os.path.split(json_filename)
    fixed_json_filename = os.path.join(directory, "fixed_" + original_filename)

    with open(fixed_json_filename, "w") as fixed_file:
        fixed_file.write(fixed_content)

    return fixed_json_filename


json_filename = "path/to/SAM2_mask_data_batch_1.json"
fixed_json_filename = fix_json_backslashes(json_filename)

print(f"Fixed JSON saved as: {fixed_json_filename}")

### Visual Inspection of SAM2 Masks

This section converts saved SAM2 masks into image overlays for visual inspection. Each mask is drawn on top of the original RGB image so segmentation quality can be checked before proceeding to the diameter estimation steps.

## Create a folder of SAM mask overlays

In [ ]:
import cv2
import numpy as np
import os
import json
from tqdm import tqdm

# Define directories
images_directory = "path/to/image_folder"
json_file_path = os.path.join(images_directory, "SAM2_mask_data_batch_0.json")
output_directory = os.path.join(images_directory, "masked_output")

# Create output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Load mask data from JSON file
with open(json_file_path, "r") as f:
    mask_data = json.load(f)

# Iterate over images and overlay masks
for image_name, mask_list in tqdm(mask_data.items(), desc="Processing Images"):
    image_path = os.path.join(images_directory, image_name)

    image = cv2.imread(image_path)
    if image is None:
        print(f"Skipping {image_name}, could not load.")
        continue

    if not mask_list:
        print(f"Skipping {image_name}, no mask found in JSON.")
        continue

    mask = np.array(mask_list, dtype=np.uint8)

    if mask.size == 0:
        print(f"Skipping {image_name}, mask is empty.")
        continue

    mask_colored = np.zeros_like(image)
    mask_colored[:, :, 2] = mask

    alpha = 0.5
    overlay = cv2.addWeighted(image, 1, mask_colored, alpha, 0)

    output_path = os.path.join(output_directory, f"masked_{image_name}")
    cv2.imwrite(output_path, overlay)

print(f"Processed images saved in: {output_directory}")